### Create KG from unstructured data using LLM and neo4j

In [5]:
# Import libraries
import wikipediaapi
import openai
from neo4j import GraphDatabase
import re
from dotenv import load_dotenv
import os

In [4]:
load_dotenv()

True

In [ ]:
### Step 1:  Ser OpenAI API key for GPT bases LLM
openai.api_key = os.getenv("OPENAI_API_KEY")

In [18]:
### Step 2: Function to extract entities from text using OpenAI GPT LLM
def extract_entities(text):
    prompt=f"Extract entities and their types from the following text:\n\n{text}\n\nFormat: Entity - Type (e.g., John Doe - Person)"
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages = [{"role":"user", "content":prompt}]
    )
    
    return response['choices'][0]['message']['content']

In [19]:
### Step 3 : Connect to Neo4j Sandbox Cloud
uri = os.getenv("bolt_url")
username = os.getenv("username")
password = os.getenv("password")
driver = GraphDatabase.driver(uri, auth=(username, password))

In [30]:
# Function to create nodes in Neo4j
def create_node(tx, label, name):
    query = f"MERGE (n:{label} {{name:$name}})"
    tx.run(query, name=name)

In [37]:
# Function to create relationship in Neo4j
def create_relationship(tx, entity1, entity2, relationship):
    query = (
        f"MATCH (a {{name:$entity1}}), (b {{name:$entity2}})"
        f"MERGE (a)-[r:{relationship}]->(b)"
        "RETURN type(r)"
    )
    tx.run(query, entity1=entity1, entity2=entity2)

In [38]:
import wikipediaapi

# step5 Fetch wikipedi apage data

wiki =wikipediaapi.Wikipedia(
    language='en',
    user_agent='my_bot (abc@example.com)'
)
page_title = "Artificial Intelligence"
page = wiki.page(page_title)

if page.exists():
    wiki_text = page.text
    print(f"Text from {page_title}: \n", wiki_text[:1000]) # preview first 1000 characters
else:
    print(f"The page '{page_title}' does not exist")

Text from Artificial Intelligence: 
 Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into general ap

In [39]:
# Step6: Extract entities from Wikipedia text
entities_text = extract_entities(wiki_text[:2000])
print("Extracted Entities:\n", entities_text)

Extracted Entities:
 1. Artificial intelligence - Concept  
2. AI - Concept  
3. computational systems - Concept  
4. human intelligence - Concept  
5. computer science - Field  
6. Google Search - Application  
7. YouTube - Application  
8. Amazon - Company  
9. Netflix - Company  
10. Google Assistant - Application  
11. Siri - Application  
12. Alexa - Application  
13. Waymo - Company  
14. chess - Game  
15. Go - Game  
16. AI research - Field  
17. knowledge representation - Concept  
18. natural language processing - Concept  
19. OpenAI - Company  
20. Google DeepMind - Company  
21. Meta - Company  
22. artificial general intelligence (AGI) - Concept  
23. psychology - Field  
24. linguistics - Field  
25. philosophy - Field  
26. neuroscience - Field  
27. operations research - Field  
28. statistics - Field  
29. mathematics - Field  


- from this list, we need toremove numbers, periods, and any special characters,
- This iteration create the list of tuples, which provides details about each company use AI in what area

In [40]:
# Parse and create relationship
import re

# Function to remove leading numbers, periods, and spaces
def clean_entity_name(entity_name):
    return re.sub(r"^\d+\.\s*", "", entity_name).strip()

def parse_entities(entities_text):
    entity_pattern = r"(.+) - (.+)"
    entities = []
    for line in entities_text.split('\n'):
        match = re.match(entity_pattern, line)
        if match:
            entity_name, entity_type = match.groups()
            entity_name = clean_entity_name(entity_name)
            entities.append((entity_name, entity_type.strip()))
    return entities

entities = parse_entities(entities_text)

In [41]:
entities

[('Artificial intelligence', 'Concept'),
 ('AI', 'Concept'),
 ('computational systems', 'Concept'),
 ('human intelligence', 'Concept'),
 ('computer science', 'Field'),
 ('Google Search', 'Application'),
 ('YouTube', 'Application'),
 ('Amazon', 'Company'),
 ('Netflix', 'Company'),
 ('Google Assistant', 'Application'),
 ('Siri', 'Application'),
 ('Alexa', 'Application'),
 ('Waymo', 'Company'),
 ('chess', 'Game'),
 ('Go', 'Game'),
 ('AI research', 'Field'),
 ('knowledge representation', 'Concept'),
 ('natural language processing', 'Concept'),
 ('OpenAI', 'Company'),
 ('Google DeepMind', 'Company'),
 ('Meta', 'Company'),
 ('artificial general intelligence (AGI)', 'Concept'),
 ('psychology', 'Field'),
 ('linguistics', 'Field'),
 ('philosophy', 'Field'),
 ('neuroscience', 'Field'),
 ('operations research', 'Field'),
 ('statistics', 'Field'),
 ('mathematics', 'Field')]

In [42]:
### Create nodes and relationships in neo4j
with driver.session() as session:
    for i, (entity1_name, entity1_type) in enumerate(entities):
        # Create nodes for the first entity
        session.execute_write(create_node, entity1_type, entity1_name)
        
        for j, (entity2_name, entity2_type) in enumerate(entities):
            if i != j:
                # Define relationships based on entity types
                if entity1_type == "Application" and entity2_type == "Concept":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "USES")
                elif entity1_type == "Application" and entity2_type == "Goal":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "ACHIEVES")
                elif entity1_type == "Application" and entity2_type == "Technique":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "IMPLEMENTED_WITH")
                elif entity1_type == "Game" and entity2_type == "Concept":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "BASED_ON")
                elif entity1_type == "Field" and entity2_type == "Concept":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "STUDIES")
                elif entity1_type == "Field" and entity2_type == "Goal":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "SUPPORTS")
                elif entity1_type == "Concept" and entity2_type == "Field":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "APPLIES_TO")
                elif entity1_type == "Year" and entity2_type == "Concept":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "INTRODUCED_IN")
                elif entity1_type == "Goal" and entity2_type == "Technique":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "ACHIEVED_BY")
                elif entity1_type == "Concept" and entity2_type == "Technique":
                    session.execute_write(create_relationship, entity1_name, entity2_name, "ENABLED_BY")

                    